#Create Spark Session

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("Customer_Behaviour") \
    .getOrCreate()

In [3]:
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=False)

# Ingest and Clean the Data

In [4]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- category: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)



In [5]:
orders_df.show(10, truncate=False)

+-----------+-----------+-----------+-----------+-----------+-------+----------+---------+
|order_id   |customer_id|city       |category   |product    |amount |order_date|status   |
+-----------+-----------+-----------+-----------+-----------+-------+----------+---------+
|ORD00000000|C000000    | hyderabad | grocery   |Oil        |invalid|01/01/2024|Cancelled|
|ORD00000001|C000001    |Pune       |Grocery    |Sugar      |35430  |2024-01-02|Completed|
|ORD00000002|C000002    |Pune       |Electronics|Mobile     |65358  |2024-01-03|Completed|
|ORD00000003|C000003    |Bangalore  |Electronics|Laptop     |5558   |2024-01-04|Completed|
|ORD00000004|C000004    |Pune       |Home       |AirPurifier|33659  |2024-01-05|Completed|
|ORD00000005|C000005    |Delhi      |Fashion    |Jeans      |8521   |2024-01-06|Completed|
|ORD00000006|C000006    |Delhi      |Grocery    |Sugar      |42383  |2024-01-07|Completed|
|ORD00000007|C000007    |Pune       |Grocery    |Rice       |45362  |2024-01-08|Completed|

# Trim text columns

In [6]:
from pyspark.sql import functions as F

cols_to_trim = ["city", "category", "product"]

cleaned_df = orders_df
for c in cols_to_trim:
    cleaned_df = cleaned_df.withColumn(c, F.trim(F.col(c)))

#Normalize city, category, product; remove duplicates and also keep only Completed ones

In [7]:
from pyspark.sql import functions as F

standardized_df = (
    cleaned_df
    .withColumn("city", F.initcap(F.col("city")))
    .withColumn("category", F.initcap(F.col("category")))
    .withColumn("product", F.initcap(F.col("product")))
)

standardized_df.select("city", "category", "product").show(20, truncate=False)

+---------+-----------+-----------+
|city     |category   |product    |
+---------+-----------+-----------+
|Hyderabad|Grocery    |Oil        |
|Pune     |Grocery    |Sugar      |
|Pune     |Electronics|Mobile     |
|Bangalore|Electronics|Laptop     |
|Pune     |Home       |Airpurifier|
|Delhi    |Fashion    |Jeans      |
|Delhi    |Grocery    |Sugar      |
|Pune     |Grocery    |Rice       |
|Bangalore|Fashion    |Jeans      |
|Kolkata  |Electronics|Laptop     |
|Bangalore|Grocery    |Sugar      |
|Kolkata  |Electronics|Tablet     |
|Bangalore|Grocery    |Sugar      |
|Pune     |Fashion    |Tshirt     |
|Mumbai   |Electronics|Tablet     |
|Pune     |Electronics|Mobile     |
|Mumbai   |Home       |Mixer      |
|Bangalore|Grocery    |Oil        |
|Kolkata  |Fashion    |Jeans      |
|Mumbai   |Electronics|Mobile     |
+---------+-----------+-----------+
only showing top 20 rows


In [8]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

amount_norm = F.regexp_replace(F.trim(F.col("amount")), r",", "")

tmp_df = (
    standardized_df
    .withColumn("amount_norm",
                F.when(F.length(amount_norm) == 0, F.lit(None)).otherwise(amount_norm))
)

In [9]:
valid_int = F.col("amount_norm").rlike(r"^[+-]?\d+$")

cleaned_amount_df = (
    tmp_df
    .withColumn(
        "amount",
        F.when(valid_int, F.col("amount_norm").cast(IntegerType()))
         .otherwise(F.lit(None).cast(IntegerType()))
    )
    .drop("amount_norm")
)

In [10]:
cleaned_amount_df.select("amount").summary().show()
cleaned_amount_df.filter(F.col("amount").isNull()).select("order_id", "amount").show(20, truncate=False)

+-------+-----------------+
|summary|           amount|
+-------+-----------------+
|  count|           274836|
|   mean|43787.37508550554|
| stddev|26192.67426878039|
|    min|              500|
|    25%|            19732|
|    50%|            43117|
|    75%|            66707|
|    max|            90000|
+-------+-----------------+

+-----------+------+
|order_id   |amount|
+-----------+------+
|ORD00000000|NULL  |
|ORD00000019|NULL  |
|ORD00000029|NULL  |
|ORD00000038|NULL  |
|ORD00000057|NULL  |
|ORD00000058|NULL  |
|ORD00000076|NULL  |
|ORD00000087|NULL  |
|ORD00000095|NULL  |
|ORD00000114|NULL  |
|ORD00000116|NULL  |
|ORD00000133|NULL  |
|ORD00000145|NULL  |
|ORD00000152|NULL  |
|ORD00000171|NULL  |
|ORD00000174|NULL  |
|ORD00000190|NULL  |
|ORD00000203|NULL  |
|ORD00000209|NULL  |
|ORD00000228|NULL  |
+-----------+------+
only showing top 20 rows


# Parse order_date into DateType → order_date_clean

In [11]:
from pyspark.sql import functions as F
date_str = F.trim(F.col("order_date"))

ts1 = F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")
ts2 = F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')")
ts3 = F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")

order_date_clean = F.to_date(F.coalesce(ts1, ts2, ts3))

final_df = cleaned_amount_df.withColumn("order_date_clean", order_date_clean)

final_df.select("order_id", "order_date", "order_date_clean").show(20, truncate=False)
final_df.printSchema()

failed = final_df.filter(F.col("order_date").isNotNull() & F.col("order_date_clean").isNull()).count()
print(f"Unparseable order_date values: {failed}")


+-----------+----------+----------------+
|order_id   |order_date|order_date_clean|
+-----------+----------+----------------+
|ORD00000000|01/01/2024|2024-01-01      |
|ORD00000001|2024-01-02|2024-01-02      |
|ORD00000002|2024-01-03|2024-01-03      |
|ORD00000003|2024-01-04|2024-01-04      |
|ORD00000004|2024-01-05|2024-01-05      |
|ORD00000005|2024-01-06|2024-01-06      |
|ORD00000006|2024-01-07|2024-01-07      |
|ORD00000007|2024-01-08|2024-01-08      |
|ORD00000008|2024-01-09|2024-01-09      |
|ORD00000009|2024-01-10|2024-01-10      |
|ORD00000010|2024-01-11|2024-01-11      |
|ORD00000011|12/01/2024|2024-01-12      |
|ORD00000012|2024-01-13|2024-01-13      |
|ORD00000013|2024/01/14|2024-01-14      |
|ORD00000014|2024-01-15|2024-01-15      |
|ORD00000015|2024-01-16|2024-01-16      |
|ORD00000016|2024-01-17|2024-01-17      |
|ORD00000017|2024-01-18|2024-01-18      |
|ORD00000018|2024-01-19|2024-01-19      |
|ORD00000019|2024-01-20|2024-01-20      |
+-----------+----------+----------

# Total number of orders.

In [14]:
total_orders = cleaned_amount_df.count()
print(f"Total number of orders: {total_orders}")


Total number of orders: 300000


# Total spending

In [16]:

from pyspark.sql.functions import sum

total_spending = cleaned_amount_df.agg(sum("amount")).first()[0]


# Average order value


In [18]:

from pyspark.sql.functions import avg

average_order_value = cleaned_amount_df.agg(avg("amount")).first()[0]


In [19]:
df=cleaned_amount_df

In [24]:

from pyspark.sql.functions import col, to_date, coalesce

df2 = df.withColumn(
    "order_date_clean",
    coalesce(
        to_date(col("order_date"), "yyyy-MM-dd"),
        to_date(col("order_date"), "dd/MM/yyyy"),
        to_date(col("order_date"), "yyyy/MM/dd")
    )
)

df_clean = df2.filter(col("order_date_clean").isNotNull())


# Number of distinct cities ordered from


In [27]:
distinct_cities = df.select("city").distinct().count()

In [29]:
distinct_cities

7

# Number of distinct categories ordered from

In [30]:
df.select("category").distinct().count()

4

# Customer Segment

In [31]:

from pyspark.sql.functions import sum, count, col

customer_agg = df_clean.groupBy("customer_id").agg(
    sum("amount").alias("total_spend"),
    count("order_id").alias("total_orders")
)


In [32]:

from pyspark.sql.functions import when

customer_segmented = customer_agg.withColumn(
    "customer_segment",
    when((col("total_spend") >= 200000) & (col("total_orders") >= 5), "VIP")
    .when(col("total_spend") >= 100000, "Premium")
    .otherwise("Regular")
)


#Window functions

In [34]:

from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, rank, dense_rank


In [35]:

from pyspark.sql.functions import sum

customer_spend = df_clean.groupBy("customer_id", "city").agg(
    sum("amount").alias("total_spend")
)


In [36]:

from pyspark.sql.window import Window
from pyspark.sql.functions import rank

w_overall = Window.orderBy(col("total_spend").desc())

customer_rank_overall = customer_spend.withColumn(
    "overall_rank",
    rank().over(w_overall)
)


In [37]:

w_city = Window.partitionBy("city").orderBy(col("total_spend").desc())

customer_rank_city = customer_spend.withColumn(
    "city_rank",
    rank().over(w_city)
)


In [38]:
top3_per_city = customer_rank_city.filter(col("city_rank") <= 3)

In [41]:
top3_per_city

DataFrame[customer_id: string, city: string, total_spend: bigint, city_rank: int]